# ANNNI finite-window gap certificates at Colab scale

This notebook independently reruns the tracked sparse-matrix experiment for Hilbert spaces up to dimension $2^{16}$. It records the physical gap from `eigsh` and compares it with the edge inferred only from Euclidean correlation moments.

In [ ]:
!rm -rf /content/aqft-split-inclusion-series
!git clone --quiet --depth 1 --branch agent/positive-spectral-resource-boundaries https://github.com/lluiseriksson/aqft-split-inclusion-series.git /content/aqft-split-inclusion-series
%cd /content/aqft-split-inclusion-series
!git rev-parse HEAD

In [ ]:
!python verification/finite_window_gap_certificates/spin_chain_experiment.py --lengths 12 14 16 --max-degree 6 --output /content/annni_colab_results.json > /content/annni_colab_stdout.txt
print('large run complete')

In [ ]:
import hashlib, json, platform, scipy, numpy as np
payload = json.load(open('/content/annni_colab_results.json'))
raw = open('/content/annni_colab_results.json', 'rb').read()
print('python', platform.python_version(), 'numpy', np.__version__, 'scipy', scipy.__version__)
print('sha256', hashlib.sha256(raw).hexdigest())
for result in payload['results']:
    rows = result['finite_window']
    first_negative = next((row['degree'] for row in rows if row['block_false_gap_localizer_min'] < 0), None)
    first_x_negative = next((row['degree'] for row in rows if row['x_only_false_gap_localizer_min'] < 0), None)
    overlap = result['probe_overlap_weights']['rows_are_excited_states_1_to_5'][0]
    print(json.dumps({
        'L': result['length'],
        'dimension': 2 ** result['length'],
        'true_gap': result['true_gap'],
        'true_edge': result['true_transfer_edge'],
        'X_overlap_first': overlap[0],
        'Z_overlap_first': overlap[1],
        'block_N6_edge': rows[-1]['block_ritz'],
        'block_N6_abs_error': abs(rows[-1]['block_ritz'] - result['true_transfer_edge']),
        'block_first_false_gap_witness_degree': first_negative,
        'X_first_false_gap_witness_degree': first_x_negative
    }, sort_keys=True))